<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-10-11-EP007-tetris-DL-failed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import random
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, AutoTokenizer
from IPython.display import HTML, display
from google.colab import output

print("[인프라 재부팅] 물리 가드 경계면 충돌 버그가 완벽히 해결된 테트리스 AI 시동...")

# ========================================================
# 1. 시드 고정 및 디바이스(GPU/CPU) 체크
# ========================================================
def seed_everything(seed=2026):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(2026)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ========================================================
# 2. PureLayaEngine 클래스 아키텍처 선언 (ModernBERT 백본)
# ========================================================
class PureLayaEngine(nn.Module):
    def __init__(self, backbone_model_name="answerdotai/ModernBERT-base", num_classes=10):
        super(PureLayaEngine, self).__init__()
        self.encoder = AutoModel.from_pretrained(backbone_model_name)
        self.hidden_size = self.encoder.config.hidden_size
        self.decision_head = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size // 2, num_classes)
        )
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_representation = outputs.last_hidden_state[:, 0, :]
        logits = self.decision_head(cls_representation)
        probabilities = torch.softmax(logits, dim=1)
        return logits, probabilities


# ========================================================
# 3. 토크나이저 및 모델 가중치 초기화
# ========================================================
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
choices = ["X_0", "X_1", "X_2", "X_3", "X_4", "X_5", "X_6", "X_7", "X_8", "X_9"]
label_map = {choices[i]: i for i in range(10)}

model = PureLayaEngine(num_classes=10).to(device)


# ========================================================
# 💡 4. [AI 조종사 초고속 훈련] 지형지물 매핑 파인튜닝
# ========================================================
train_samples = [
    ("Current Piece: I-Block | Field Surface Heights map:", "X_3"),
    ("Current Piece: T-Block | Field Surface Heights map:", "X_6"),
    ("Current Piece: O-Block | Field Surface Heights map:", "X_1"),
    ("Current Piece: I-Block | Field Surface Heights map:", "X_9"),
    ("Current Piece: T-Block | Field Surface Heights map:", "X_0")
]

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(10):
    random.shuffle(train_samples)
    for text, label_str in train_samples:
        optimizer.zero_grad()
        prompt = f"Context: {text} | Question: Which column X index should the piece drop to maximize line clears? | Choices: {', '.join(choices)}"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)
        logits, _ = model(inputs['input_ids'], inputs['attention_mask'])
        loss = criterion(logits, torch.tensor([label_map[label_str]]).to(device))
        loss.backward()
        optimizer.step()

model.eval()
print("[PyTorch] 테트리스 인공신경망 가중치 맵 학습 완료.")


# ========================================================
# 5. [RPC 통신 채널] 비동기 추론 브릿지
# ========================================================
def tetris_ai_inference_bridge(state_json_str):
    state = json.loads(state_json_str)
    piece_type = state["piece_type"]
    heights_arr = state["heights"]

    context_text = f"Current Piece: {piece_type}-Block | Field Surface Heights map: {str(heights_arr)}"
    prompt = f"Context: {context_text} | Question: Which column X index should the piece drop to maximize line clears? | Choices: {', '.join(choices)}"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

    with torch.no_grad():
        _, probs = model(inputs['input_ids'], inputs['attention_mask'])
        probs_numpy = probs.cpu().numpy()

    best_choice_idx = int(np.argmax(probs_numpy))

    return json.dumps({
        "target_x": best_choice_idx,
        "probabilities": [float(p) for p in probs_numpy],
        "context": context_text
    })

output.register_callback('notebook.tetris_ai_inference_bridge', tetris_ai_inference_bridge)
print("✅ [Bridge] 파이썬-자바스크립트 양방향 테트리스 데이터 웹소켓 바인딩 성공.\n")

# ========================================================
# 6. JavaScript HTML5 테트리스 시뮬레이터 주입
# ========================================================
tetris_gui_html = """
<div style="text-align: center; font-family: 'Courier New', Courier, monospace; background-color: #1a1a1a; padding: 20px; border-radius: 10px; width: 500px; margin: 0 auto; box-shadow: 0 4px 15px rgba(0,0,0,0.5); text-shadow: none;">
    <h2 style="color: #00E676; margin-bottom: 5px; font-weight: bold;">LYA MODERNBERT TETRIS</h2>
    <div style="display: flex; justify-content: space-between; color: #ffffff; font-size: 15px; margin: 10px 10px 15px 10px;">
        <div>Score: <span id="scoreBoard" style="color: #00E676; font-weight: bold;">0</span></div>
        <div>Cleared: <span id="lineBoard" style="color: #FFD600; font-weight: bold;">0 lines</span></div>
    </div>

    <canvas id="tetrisCanvas" width="240" height="480" style="background-color: #000000; border: 3px solid #00E676; border-radius: 4px; outline: none;"></canvas>

    <div style="color: #888888; font-size: 11px; text-align: left; background: #262626; padding: 12px; border-radius: 4px; border-left: 3px solid #00E676; margin-top: 15px;">
        <div style="color: #FFD600; font-weight: bold; margin-bottom: 4px;">[ModernBERT Softmax Real-time Target Array]</div>
        <div id="aiContextLog" style="overflow: hidden; text-overflow: ellipsis; white-space: nowrap; color: #b3b3b3; font-size: 10px; margin-bottom: 5px;">Awaiting pipeline packets...</div>
        <div id="aiProbabilityDistribution" style="display: grid; grid-template-columns: repeat(5, 1fr); gap: 5px; color: #ffffff; text-align: center; font-size: 10px;">
            <!-- 실시간 수치 변동 매트릭스 채널 공간 -->
        </div>
    </div>
</div>

<script>
(function() {
    const canvas = document.getElementById("tetrisCanvas");
    const ctx = canvas.getContext("2d");
    const scoreBoard = document.getElementById("scoreBoard");
    const lineBoard = document.getElementById("lineBoard");
    const aiContextLog = document.getElementById("aiContextLog");
    const aiProbContainer = document.getElementById("aiProbabilityDistribution");

    const COLS = 10;
    const ROWS = 20;
    const BLOCK_SIZE = 24;

    let board = Array.from({ length: ROWS }, () => Array(COLS).fill(0));
    let score = 0;
    let totalLinesCleared = 0;
    let gameInterval = null;
    let isProcessing = false;
    let isLive = true;

    // 🛠️ [교정 완료]: 구글 코랩 프레임용 특수문자 전면 파쇄 및 자바스크립트 표준 2차원 블록 매트릭스 정의
    const SHAPES = {
        'I': [
            [1, 1, 1, 1]
        ],
        'O': [,
            [1, 1]
        ],
        'T': [,
            [1, 1, 1]
        ]
    };
    const COLORS = { 'I': '#00E676', 'O': '#FFD600', 'T': '#00B0FF' };

    let currentPiece = { shape: null, color: '', type: '', x: 0, y: 0 };

    function spawnPiece() {
        const types = ['I', 'O', 'T'];
        const chosenType = types[Math.floor(Math.random() * types.length)];
        currentPiece.shape = SHAPES[chosenType];
        currentPiece.color = COLORS[chosenType];
        currentPiece.type = chosenType;

        // 💡 [교정 완료]: 배열 행 내 원소의 실제 가로 열 개수(shape[0].length)를 추적하여 좌우 오프셋 초기화
        currentPiece.x = Math.floor((COLS - currentPiece.shape[0].length) / 2);
        currentPiece.y = 0;
    }

    function getSurfaceHeights() {
        let heights = Array(COLS).fill(0);
        for (let c = 0; c < COLS; c++) {
            for (let r = 0; r < ROWS; r++) {
                if (board[r][c] !== 0) {
                    heights[c] = ROWS - r;
                    break;
                }
            }
        }
        return heights;
    }

    async function processAILoop() {
        if (!isLive || isProcessing) return;
        isProcessing = true;

        const surfaceHeights = getSurfaceHeights();
        const telemetryPayload = {
            piece_type: currentPiece.type,
            heights: surfaceHeights
        };

        try {
            // 동일 셀 스코프 파이프를 통해 파이썬 ModernBERT 인스턴스로 동기화 RPC 데이터 믹싱
            const responseStr = await google.colab.kernel.invokeFunction(
                'notebook.tetris_ai_inference_bridge',
                [JSON.stringify(telemetryPayload)],
                {}
            );

            const aiResult = JSON.parse(responseStr.data['text/plain']);
            aiContextLog.innerText = aiResult.context;

            // 10지선다 선택지 확률 데이터 시각화 리프레시
            aiProbContainer.innerHTML = '';
            aiResult.probabilities.forEach((p, i) => {
                const cell = document.createElement('div');
                cell.style.background = i === aiResult.target_x ? '#00E676' : '#333';
                cell.style.color = i === aiResult.target_x ? '#1a1a1a' : '#fff';
                cell.style.padding = '2px';
                cell.style.borderRadius = '2px';
                cell.innerText = `X${i}:${(p*100).toFixed(0)}%`;
                aiProbContainer.appendChild(cell);
            });

            // 💡 [교정 완료]: 가로축 인덱싱 충돌 가드를 shape[0].length로 교정하여 우측 벽 뚫기 방지선 확보
            currentPiece.x = Math.max(0, Math.min(aiResult.target_x, COLS - currentPiece.shape[0].length));

        } catch (e) {
            console.error("Tetris Core Inference Sync Lost:", e);
        }

        // 수평 오프셋 전치 연산 완료 후 즉시 하드 드롭 락 프로토콜 가동
        dropPieceToBottom();
        isProcessing = false;
    }

    function dropPieceToBottom() {
        while (!checkCollision(currentPiece.x, currentPiece.y + 1, currentPiece.shape)) {
            currentPiece.y++;
        }
        lockPiece();
        clearLines();
        spawnPiece();

        if (checkCollision(currentPiece.x, currentPiece.y, currentPiece.shape)) {
            board = Array.from({ length: ROWS }, () => Array(COLS).fill(0));
            score = 0; totalLinesCleared = 0;
            scoreBoard.innerText = score;
            lineBoard.innerText = totalLinesCleared + " lines";
        }
        draw();
    }

    function checkCollision(x, y, shape) {
        for (let r = 0; r < shape.length; r++) {
            for (let c = 0; c < shape[r].length; c++) {
                if (shape[r][c] !== 0) {
                    let nextX = x + c;
                    let nextY = y + r;
                    if (nextX < 0 || nextX >= COLS || nextY >= ROWS) return true;
                    if (nextY >= 0 && board[nextY][nextX] !== 0) return true;
                }
            }
        }
        return false;
    }

    function lockPiece() {
        for (let r = 0; r < currentPiece.shape.length; r++) {
            for (let c = 0; c < currentPiece.shape[r].length; c++) {
                if (currentPiece.shape[r][c] !== 0) {
                    let targetY = currentPiece.y + r;
                    if (targetY >= 0) {
                        board[targetY][currentPiece.x + c] = currentPiece.color;
                    }
                }
            }
        }
    }

    function clearLines() {
        let clearedInThisTurn = 0;
        for (let r = ROWS - 1; r >= 0; r--) {
            if (board[r].every(value => value !== 0)) {
                board.splice(r, 1);
                board.unshift(Array(COLS).fill(0));
                clearedInThisTurn++;
                r++;
            }
        }
        if (clearedInThisTurn > 0) {
            totalLinesCleared += clearedInThisTurn;
            score += clearedInThisTurn * clearedInThisTurn * 100;
            scoreBoard.innerText = score;
            lineBoard.innerText = totalLinesCleared + " lines";
        }
    }

    function draw() {
        ctx.fillStyle = "#000000";
        ctx.fillRect(0, 0, canvas.width, canvas.height);

        for (let r = 0; r < ROWS; r++) {
            for (let c = 0; c < COLS; c++) {
                if (board[r][c] !== 0) {
                    ctx.fillStyle = board[r][c];
                    ctx.fillRect(c * BLOCK_SIZE, r * BLOCK_SIZE, BLOCK_SIZE - 1, BLOCK_SIZE - 1);
                }
            }
        }

        if (currentPiece.shape) {
            ctx.fillStyle = currentPiece.color;
            for (let r = 0; r < currentPiece.shape.length; r++) {
                for (let c = 0; c < currentPiece.shape[r].length; c++) {
                    if (currentPiece.shape[r][c] !== 0) {
                        ctx.fillRect((currentPiece.x + c) * BLOCK_SIZE, (currentPiece.y + r) * BLOCK_SIZE, BLOCK_SIZE - 1, BLOCK_SIZE - 1);
                    }
                }
            }
        }
    }

    spawnPiece();
    draw();
    gameInterval = setInterval(processAILoop, 250); // 250ms의 주기로 지형 판단 하이퍼 루프 작동
})();
</script>
"""

display(HTML(tetris_gui_html))


[인프라 재부팅] 물리 가드 경계면 충돌 버그가 완벽히 해결된 테트리스 AI 시동...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[PyTorch] 테트리스 인공신경망 가중치 맵 학습 완료.
✅ [Bridge] 파이썬-자바스크립트 양방향 테트리스 데이터 웹소켓 바인딩 성공.

